# CrisisSense-LLM: Classification-Guided RAG System
### Disaster Situational Awareness Chatbot

**Pipeline:**
1. User query → Fine-tuned Qwen2.5 classifier → Labels (event, humanitarian type)
2. Labels → Filter FAISS vector index → Retrieve top-K relevant tweets
3. Retrieved tweets + query → Qwen2.5 generator → Final response
4. Gradio UI displays response + source tweets

In [ ]:
# STEP 1 — Install dependencies
!pip install -q torch transformers peft accelerate sentencepiece bitsandbytes
!pip install -q faiss-gpu sentence-transformers
!pip install -q gradio langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 8.1 MB/s eta 0:00:00


In [ ]:
# STEP 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# STEP 3 — Paths
import os

DRIVE_ROOT   = "/content/drive/MyDrive/disaster_llm"
BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
BEST_CKPT    = f"{DRIVE_ROOT}/outputs/qwen25_7b_lora/checkpoint-6416"  # best checkpoint
TEST_FILE    = f"{DRIVE_ROOT}/test_v2.jsonl"             # source tweets
TRAIN_FILE   = f"{DRIVE_ROOT}/train_v2.jsonl"            # more tweets for KB
INDEX_DIR    = f"{DRIVE_ROOT}/rag/index"

os.makedirs(INDEX_DIR, exist_ok=True)

print("Checkpoint exists:", os.path.exists(BEST_CKPT))
print("Test file exists: ", os.path.exists(TEST_FILE))

Checkpoint exists: True
Test file exists:  True


In [ ]:
# STEP 4 — Load knowledge base tweets (template-4 combined records)
import json

def load_kb_tweets(jsonl_path, limit=5000):
    """
    Load template-4 records as knowledge base.
    Each record has: instruction (contains tweet text) + response (labels)
    """
    tweets = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            if obj.get("template") == 4:
                # Extract tweet text from instruction
                instruction = obj["instruction"]
                # Tweet text is after "text: " in the instruction
                if "text:" in instruction:
                    tweet_text = instruction.split("text:")[-1].split("\nResponse:")[0].strip()
                else:
                    tweet_text = instruction[-500:]  # fallback: last 500 chars

                labels = obj["response"]
                tweets.append({
                    "text": tweet_text,
                    "event_type": labels.get("Event type", "DISASTER"),
                    "useful": labels.get("Useful", "False"),
                    "humanitarian": labels.get("Humanitarian aid type", "NOT HUMANITARIAN"),
                })
                if len(tweets) >= limit:
                    break
    return tweets

print("Loading knowledge base tweets...")
kb_tweets = load_kb_tweets(TRAIN_FILE, limit=5000)
print(f"Loaded {len(kb_tweets):,} tweets into knowledge base")
print("\nSample:")
print(kb_tweets[0])

Loading knowledge base tweets...
Loaded 5,000 tweets into knowledge base

Sample:
{'text': 'I congratulate you for the good job you dou', 'event_type': 'DISASTER', 'useful': 'True', 'humanitarian': 'OTHER RELEVANT INFORMATION'}


In [ ]:
# STEP 5 — Build FAISS vector index
import numpy as np
import faiss
import pickle
import os


from sentence_transformers import SentenceTransformer

INDEX_PATH  = f"{INDEX_DIR}/tweets.index"
TWEETS_PATH = f"{INDEX_DIR}/tweets.pkl"
EMBED_MODEL = "all-MiniLM-L6-v2"  # simpler model, no conflict, works great

if os.path.exists(INDEX_PATH) and os.path.exists(TWEETS_PATH):
    print("Loading existing FAISS index...")
    index = faiss.read_index(INDEX_PATH)
    with open(TWEETS_PATH, "rb") as f:
        kb_tweets = pickle.load(f)
    print(f"Index loaded: {index.ntotal} vectors")
else:
    print("Building FAISS index from scratch...")
    embedder = SentenceTransformer(EMBED_MODEL)

    texts = [t["text"] for t in kb_tweets]
    print(f"Encoding {len(texts):,} tweets...")
    embeddings = embedder.encode(
        texts,
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True,
    )

    faiss.normalize_L2(embeddings)

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    faiss.write_index(index, INDEX_PATH)
    with open(TWEETS_PATH, "wb") as f:
        pickle.dump(kb_tweets, f)

    print(f"Index built and saved: {index.ntotal} vectors")
    del embedder

Loading existing FAISS index...
Index loaded: 5000 vectors


In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "all-MiniLM-L6-v2"

print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)
print("Embedding model loaded.")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [ ]:
import os

os.environ.pop("TRANSFORMERS_OFFLINE", None)
os.environ.pop("HF_DATASETS_OFFLINE", None)
os.environ.pop("HF_ENDPOINT", None)

'https://hf-mirror.com'

In [ ]:
!ping huggingface.co

/bin/bash: line 1: ping: command not found


In [ ]:
print(BASE_MODEL)
print(BEST_CKPT)

Qwen/Qwen2.5-7B-Instruct
/content/drive/MyDrive/disaster_llm/outputs/qwen25_7b_lora/checkpoint-6416


In [ ]:
from huggingface_hub import HfApi

try:
    api = HfApi()
    models = list(api.list_models(limit=3))
    print(" Hugging Face connection works!")
except Exception as e:
    print(" Connection failed")
    print(e)

 Hugging Face connection works!


In [ ]:
import os

print("HF_ENDPOINT =", os.environ.get("HF_ENDPOINT"))
print("TRANSFORMERS_OFFLINE =", os.environ.get("TRANSFORMERS_OFFLINE"))
print("HF_DATASETS_OFFLINE =", os.environ.get("HF_DATASETS_OFFLINE"))

HF_ENDPOINT = None
TRANSFORMERS_OFFLINE = None
HF_DATASETS_OFFLINE = None


In [ ]:
from huggingface_hub import constants

print(constants.ENDPOINT)

https://hf-mirror.com


In [ ]:
import huggingface_hub
import transformers

print("huggingface_hub:", huggingface_hub.__version__)
print("transformers:", transformers.__version__)

huggingface_hub: 1.23.0
transformers: 5.13.1


In [ ]:
import os

for k, v in os.environ.items():
    if "HF" in k.upper():
        print(k, "=", v)

In [ ]:
# STEP 6 — Load classifier + generator model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print("Loading model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Load fine-tuned LoRA adapter
model = PeftModel.from_pretrained(base_model, BEST_CKPT)
model.eval()
print("Model loaded successfully.")

Loading model...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully.


In [ ]:
# STEP 7 — Classifier function
import re

EVENT_TYPES = [
    "HURRICANE", "FLOOD", "EARTHQUAKE", "DISASTER", "EVENTS", "EXPLOSION",
    "BOMBING", "FIRE", "LANDSLIDE", "CRASH", "DISEASE", "SHOOTING",
    "COLLAPSE", "HAZARD", "VOLCANO",
]
HUMANITARIAN_TYPES = [
    "NOT HUMANITARIAN", "OTHER RELEVANT INFORMATION", "DONATION AND VOLUNTEERING",
    "REQUESTS OR NEEDS", "SYMPATHY AND SUPPORT", "INFRASTRUCTURE AND UTILITY DAMAGE",
    "AFFECTED INDIVIDUAL", "CAUTION AND ADVICE", "INJURED OR DEAD PEOPLE",
    "DISEASE RELATED", "RESPONSE EFFORTS", "PERSONAL UPDATE",
    "MISSING AND FOUND PEOPLE", "DISPLACED AND EVACUATION",
    "PHYSICAL LANDSLIDE", "TERRORISM RELATED",
]
EVENT_LIST_STR = "\n".join(EVENT_TYPES)
HUMANITARIAN_LIST_STR = "\n".join(HUMANITARIAN_TYPES)

def build_classification_prompt(text):
    return (
        "### Instruction:\n"
        "This is a multi-label classification task.\n"
        f"FIRST, classify the event type into one of:\n{EVENT_LIST_STR}\n"
        "SECOND, is the text useful for humanitarian aid? Answer True or False.\n"
        f"THIRD, classify the humanitarian aid type into one of:\n{HUMANITARIAN_LIST_STR}\n"
        "Format as JSON with keys: Event type, Useful, Humanitarian aid type\n\n"
        f"text: {text}\nResponse:"
    )

def extract_field(text, patterns):
    for pattern in patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            return m.group(1).strip()
    return None

@torch.no_grad()
def classify_query(text):
    prompt = build_classification_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    output_ids = model.generate(**inputs, max_new_tokens=100, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    event = extract_field(response, [
        r'"Event type"\s*:\s*"([^"]+)"',
        r'Event type\s*:\s*([A-Z ]+)',
    ]) or "DISASTER"

    humanitarian = extract_field(response, [
        r'"Humanitarian aid type"\s*:\s*"([^"]+)"',
        r'Humanitarian aid type\s*:\s*([A-Z &]+)',
    ]) or "OTHER RELEVANT INFORMATION"

    useful = extract_field(response, [
        r'"Useful"\s*:\s*"?(True|False)"?',
        r'Useful\s*:\s*(True|False)',
    ]) or "True"

    return {
        "Event type": event.upper().strip(),
        "Useful": useful,
        "Humanitarian aid type": humanitarian.upper().strip(),
        "raw_response": response,
    }

# Test classifier
test_result = classify_query("People are trapped in flood waters and need rescue boats immediately")
print("Classifier test:")
print(f"  Event type: {test_result['Event type']}")
print(f"  Useful: {test_result['Useful']}")
print(f"  Humanitarian: {test_result['Humanitarian aid type']}")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Classifier test:
  Event type: DISASTER
  Useful: True
  Humanitarian: REQUESTS OR NEEDS


In [ ]:
def retrieve(
    query: str,
    labels: dict,
    top_k: int = 8,
    filter_mode: str = "dual",
):
    """
    Retrieve disaster reports using Classification-Guided RAG.

    Modes:
        none  -> Standard RAG
        event -> Filter by event only
        dual  -> Filter by event + humanitarian category
    """

    # Encode query
    query_emb = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    scores, indices = index.search(query_emb, 200)

    scores = scores[0]
    indices = indices[0]

    event_type = labels.get("Event type", "").upper().strip()
    humanitarian = labels.get("Humanitarian aid type", "").upper().strip()

    retrieved = []
    seen = set()

    def add_result(idx, score):
        if idx in seen:
            return

        seen.add(idx)

        tweet = kb_tweets[idx]

        retrieved.append({
            **tweet,
            "score": float(score)
        })

    # -----------------------------
    # Stage 1
    # Dual filtering
    # -----------------------------
    if filter_mode == "dual":

        for idx, score in zip(indices, scores):

            if idx < 0 or idx >= len(kb_tweets):
                continue

            tweet = kb_tweets[idx]

            event_match = (
                event_type in tweet["event_type"].upper()
            )

            humanitarian_match = (
                humanitarian in tweet["humanitarian"].upper()
            )

            if event_match and humanitarian_match:
                add_result(idx, score)

            if len(retrieved) >= top_k:
                return retrieved

    # -----------------------------
    # Stage 2
    # Event-only fallback
    # -----------------------------
    if filter_mode in ["dual", "event"]:

        for idx, score in zip(indices, scores):

            if idx < 0 or idx >= len(kb_tweets):
                continue

            tweet = kb_tweets[idx]

            if event_type in tweet["event_type"].upper():
                add_result(idx, score)

            if len(retrieved) >= top_k:
                return retrieved

    # -----------------------------
    # Stage 3
    # Pure semantic fallback
    # -----------------------------
    for idx, score in zip(indices, scores):

        if idx < 0 or idx >= len(kb_tweets):
            continue

        add_result(idx, score)

        if len(retrieved) >= top_k:
            break

    # Final ranking
    retrieved = sorted(
        retrieved,
        key=lambda x: x["score"],
        reverse=True,
    )

    return retrieved

In [ ]:
# STEP 9 — Improved Generator
@torch.no_grad()
def generate_response(query: str, retrieved_tweets: list) -> str:
    """
    Generate a grounded response from retrieved disaster reports.
    """

    # Sort by similarity (highest first)
    retrieved_tweets = sorted(
        retrieved_tweets,
        key=lambda x: x.get("score", 0),
        reverse=True
    )

    # Build structured context
    context = "\n\n".join([
        f"""Report {i+1}
Event Type: {t['event_type']}
Humanitarian Category: {t['humanitarian']}
Similarity Score: {t.get('score', 0):.3f}

Report:
{t['text']}"""
        for i, t in enumerate(retrieved_tweets)
    ])

    prompt = f"""

You are an expert humanitarian disaster analyst.

Answer the user's question ONLY using the retrieved evidence.

Rules:

- Do not invent information.
- Do not use outside knowledge.
- Do not generate hashtags.
- Do not generate citations like [1] or markdown references.
- Summarize information across all reports.
- Keep the answer under 150 words.
- If the reports do not contain enough information, clearly state that.

Retrieved Evidence:

{context}

Question:
{query}

Answer:
"""


    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output_ids = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        temperature=None,
        top_p=None,
        repetition_penalty=1.5,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

In [ ]:
# STEP 10 — Full RAG pipeline
def rag_pipeline(query: str, top_k: int = 5):
    """
    Full pipeline:
    1. Classify query
    2. Retrieve filtered tweets
    3. Generate response
    """
    # Step 1: Classify
    labels = classify_query(query)

    # Step 2: Retrieve
    retrieved = retrieve(query, labels, top_k=top_k)

    # Step 3: Generate
    response = generate_response(query, retrieved)

    return {
        "response": response,
        "labels": labels,
        "retrieved_tweets": retrieved,
    }

# Test full pipeline
print("Testing full RAG pipeline...")
result = rag_pipeline("What kind of help do people need after the flood?")
print(f"\nQuery classified as:")
print(f"  Event: {result['labels']['Event type']}")
print(f"  Humanitarian: {result['labels']['Humanitarian aid type']}")
print(f"\nRetrieved {len(result['retrieved_tweets'])} tweets")
print(f"\nGenerated response:")
print(result['response'])

Testing full RAG pipeline...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Query classified as:
  Event: DISASTER
  Humanitarian: REQUESTS OR NEEDS

Retrieved 5 tweets

Generated response:
People require various forms of support following flooding events including shelter (like tents), basic supplies needed during emergencies (food items e.g., posho & beans) along with water treatment solutions against diseases spread via contaminated waters due mosquitoes among others . Additionally they seek rebuilding materials especially if homes were destroyed completely while some may move from rural areas towards urban centers seeking safety thus necessitating further social services intervention too ensuring continuity access healthcare educational opportunities etc.. Financial interventions also play crucial role helping farmers recover lost crops / land alongside facilitating loan restructuring programs aimed at alleviating economic hardships faced post-disaster conditions affecting livelihoods directly impacted


# RAG

In [ ]:
# STEP 8 — Retriever (بدون reranker، bi-encoder score)
import torch
import numpy as np
import time

def retrieve(
    query: str,
    labels: dict,
    top_k: int = 5,
    filter_mode: str = "dual",
    candidate_pool: int = 200,
):
    """
    Classification-Guided RAG Retriever.
    Stages:
        1. Dense retrieval via FAISS
        2. Label-based filtering (3 levels)
        3. Sort by similarity score
    """

    # Stage 1: Dense retrieval
    query_emb = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    scores, indices = index.search(query_emb, candidate_pool)
    scores  = scores[0]
    indices = indices[0]

    event_type   = labels.get("Event type", "").upper().strip()
    humanitarian = labels.get("Humanitarian aid type", "").upper().strip()

    candidates = []
    seen = set()

    def try_add(idx, score):
        if idx < 0 or idx >= len(kb_tweets) or idx in seen:
            return
        seen.add(idx)
        candidates.append({**kb_tweets[idx], "score": float(score)})

    # Level 1: dual filter (event + humanitarian)
    if filter_mode == "dual":
        for idx, score in zip(indices, scores):
            if idx < 0 or idx >= len(kb_tweets): continue
            t = kb_tweets[idx]
            if (event_type in t["event_type"].upper() and
                humanitarian in t["humanitarian"].upper()):
                try_add(idx, score)
            if len(candidates) >= top_k: break

    # Level 2: event-only fallback
    if filter_mode in ["dual", "event"]:
        for idx, score in zip(indices, scores):
            if idx < 0 or idx >= len(kb_tweets): continue
            t = kb_tweets[idx]
            if event_type in t["event_type"].upper():
                try_add(idx, score)
            if len(candidates) >= top_k: break

    # Level 3: pure semantic fallback (standard RAG)
    for idx, score in zip(indices, scores):
        try_add(idx, score)
        if len(candidates) >= top_k: break

    return sorted(candidates, key=lambda x: x["score"], reverse=True)[:top_k]


# STEP 9 — Generator
@torch.no_grad()
def generate_response(query: str, retrieved_tweets: list) -> str:
    if not retrieved_tweets:
        return "I could not find relevant disaster reports to answer your question."

    # Deduplicate
    seen_texts = set()
    unique_tweets = []
    for t in retrieved_tweets:
        key = t["text"][:50].lower()
        if key not in seen_texts:
            seen_texts.add(key)
            unique_tweets.append(t)

    context = "\n\n".join([
        f"[Report {i+1} | {t['event_type']} | {t['humanitarian']} | "
        f"Relevance: {t.get('score', 0):.3f}]\n{t['text']}"
        for i, t in enumerate(unique_tweets)
    ])

    prompt = f"""You are an expert humanitarian disaster analyst providing situational awareness.

RETRIEVED DISASTER REPORTS:
{context}

STRICT RULES:
- Answer ONLY based on the retrieved reports above
- Do NOT invent facts or use outside knowledge
- Do NOT add hashtags, citations, or markdown
- Be concise, factual, and helpful
- If reports lack sufficient information, clearly state that
- Maximum 150 words

QUESTION: {query}

ANSWER:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output_ids = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip()

    for marker in ["ANSWER:", "Answer:", "QUESTION:", "RETRIEVED"]:
        if marker in response:
            response = response.split(marker)[-1].strip()

    return response


# STEP 10 — Full RAG Pipeline with timing
def rag_pipeline(query: str, top_k: int = 5, filter_mode: str = "dual"):
    timings = {}
    total_start = time.time()

    t0 = time.time()
    labels = classify_query(query)
    timings["classification_s"] = round(time.time() - t0, 3)

    t0 = time.time()
    retrieved = retrieve(query, labels, top_k=top_k, filter_mode=filter_mode)
    timings["retrieval_s"] = round(time.time() - t0, 3)

    t0 = time.time()
    response = generate_response(query, retrieved)
    timings["generation_s"] = round(time.time() - t0, 3)

    timings["total_s"] = round(time.time() - total_start, 3)

    return {
        "response":         response,
        "labels":           labels,
        "retrieved_tweets": retrieved,
        "timings":          timings,
        "filter_mode":      filter_mode,
    }


# RAG Comparison: Standard vs Event vs Dual
test_queries = [
    "What kind of help do people need after the flood?",
    "Are there missing people after the earthquake?",
    "What infrastructure was damaged in the hurricane?",
    "Where can people donate for disaster relief?",
    "What evacuation routes are available?",
]

print("=" * 70)
print("RAG COMPARISON: Standard vs Event-filtered vs Dual-filtered")
print("=" * 70)
print(f"{'Query':<45} {'Standard':>10} {'Event-only':>12} {'Dual':>8}")
print("-" * 70)

latencies = {"none": [], "event": [], "dual": []}

for query in test_queries:
    labels = classify_query(query)
    row_scores = {}

    for mode in ["none", "event", "dual"]:
        t0 = time.time()
        retrieved = retrieve(query, labels, top_k=5, filter_mode=mode)
        latencies[mode].append(time.time() - t0)
        avg_score = sum(t["score"] for t in retrieved) / len(retrieved) if retrieved else 0
        row_scores[mode] = avg_score

    print(f"{query[:44]:<45} "
          f"{row_scores['none']:>9.3f} "
          f"{row_scores['event']:>11.3f} "
          f"{row_scores['dual']:>8.3f}")

print("\nAverage Retrieval Latency:")
for mode, times in latencies.items():
    print(f"  {mode:<12}: {sum(times)/len(times)*1000:.1f}ms")

print("\nNote: Higher score = more semantically similar tweets retrieved")

# Quick test
print("\n" + "="*50)
print("FULL PIPELINE TEST")
print("="*50)
result = rag_pipeline("What kind of help do people need after the flood?")
print(f"\nClassification:")
print(f"  Event:        {result['labels']['Event type']}")
print(f"  Humanitarian: {result['labels']['Humanitarian aid type']}")
print(f"\nResponse:\n{result['response']}")
print(f"\nLatency Breakdown:")
for k, v in result['timings'].items():
    print(f"  {k}: {v}s")

RAG COMPARISON: Standard vs Event-filtered vs Dual-filtered
Query                                           Standard   Event-only     Dual
----------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


What kind of help do people need after the f      0.641       0.580    0.574
Are there missing people after the earthquak      0.586       0.572    0.555
What infrastructure was damaged in the hurri      0.633       0.632    0.621
Where can people donate for disaster relief?      0.654       0.628    0.625
What evacuation routes are available?             0.562       0.504    0.500

Average Retrieval Latency:
  none        : 11.0ms
  event       : 9.9ms
  dual        : 9.3ms

Note: Higher score = more semantically similar tweets retrieved

FULL PIPELINE TEST

Classification:
  Event:        DISASTER
  Humanitarian: REQUESTS OR NEEDS

Response:
Families require assistance such as tape lines for drying foods, tents for temporary shelter, mosquito nets, water purifiers, posho (maize flour), and beans to support their basic needs post-flooding. Additionally, there is mention of infrastructure damage where houses fell due to flooding necessitating immediate relief efforts from local authori

In [ ]:
import gradio as gr
print(gr.__version__)

6.20.0


In [ ]:
import gradio as gr
help(gr.Chatbot.__init__)

Help on function __init__ in module gradio.components.chatbot:

__init__(self, value: 'list[MessageDict | Message] | Callable | None' = None, *, label: 'str | I18nData | None' = None, every: 'Timer | float | None' = None, inputs: 'Component | Sequence[Component] | set[Component] | None' = None, show_label: 'bool | None' = None, container: 'bool' = True, scale: 'int | None' = None, min_width: 'int' = 160, visible: "bool | Literal['hidden']" = True, elem_id: 'str | None' = None, elem_classes: 'list[str] | str | None' = None, autoscroll: 'bool' = True, render: 'bool' = True, key: 'int | str | tuple[int | str, ...] | None' = None, preserved_by_key: 'list[str] | str | None' = 'value', height: 'int | str | None' = 400, resizable: 'bool' = False, max_height: 'int | str | None' = None, min_height: 'int | str | None' = None, editable: "Literal['user', 'all'] | None" = None, latex_delimiters: 'list[dict[str, str | bool]] | None' = None, rtl: 'bool' = False, buttons: "list[Literal['share', 'copy'

In [ ]:
# STEP 11 — Professional Gradio UI (fixed for Gradio 6.19.0)
import gradio as gr

def chatbot_fn(query, history, top_k):
    if not query.strip():
        return history, "", "", ""

    result = rag_pipeline(query, top_k=int(top_k))

    labels_text = (
        f"**Event Type:** {result['labels']['Event type']}\n\n"
        f"**Useful for Aid:** {result['labels']['Useful']}\n\n"
        f"**Humanitarian Type:** {result['labels']['Humanitarian aid type']}"
    )

    tweets_text = "\n\n---\n\n".join([
        f"**Tweet {i+1}** (similarity: {t['score']:.3f})\n\n"
        f"*{t['event_type']} | {t['humanitarian']}*\n\n"
        f"{t['text']}"
        for i, t in enumerate(result['retrieved_tweets'])
    ])

    history.append({"role": "user", "content": query})
    history.append({"role": "assistant", "content": result['response']})

    return history, labels_text, tweets_text, ""


with gr.Blocks(title="CrisisSense-LLM") as demo:

    gr.Markdown("""
    # 🚨 CrisisSense-LLM: Disaster Response Assistant
    ### Classification-Guided Retrieval-Augmented Generation
    Ask questions about disaster situations. The system classifies your query,
    retrieves relevant social media reports, and generates an informed response.
    """)

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(
                label="Conversation",
                height=450,
                layout="bubble",
                rtl=False,
                placeholder="Ask a question about disaster situations...",
            )
            with gr.Row():
                query_box = gr.Textbox(
                    label="Your Question",
                    placeholder="e.g. What kind of help do people need after the flood?",
                    scale=4,
                )
                submit_btn = gr.Button("Send 🚀", variant="primary", scale=1)

            top_k_slider = gr.Slider(
                minimum=1, maximum=10, value=5, step=1,
                label="Number of tweets to retrieve"
            )
            clear_btn = gr.Button("Clear Chat", variant="secondary")

        with gr.Column(scale=1):
            labels_box = gr.Markdown(value="*Classification results will appear here...*")
            gr.Markdown("---")
            tweets_box = gr.Markdown(value="*Retrieved tweets will appear here...*")

    gr.Examples(
        examples=[
            "What kind of help do people need after the flood?",
            "Are there any missing people reported after the earthquake?",
            "What infrastructure was damaged in the hurricane?",
            "Where can people donate for disaster relief?",
            "What evacuation routes are available?",
        ],
        inputs=query_box,
        label="Example Questions"
    )

    history_state = gr.State([])

    submit_btn.click(
        fn=chatbot_fn,
        inputs=[query_box, history_state, top_k_slider],
        outputs=[chatbot, labels_box, tweets_box, query_box],
    ).then(fn=lambda h: h, inputs=[chatbot], outputs=[history_state])

    query_box.submit(
        fn=chatbot_fn,
        inputs=[query_box, history_state, top_k_slider],
        outputs=[chatbot, labels_box, tweets_box, query_box],
    ).then(fn=lambda h: h, inputs=[chatbot], outputs=[history_state])

    clear_btn.click(
        fn=lambda: ([], [], "", ""),
        outputs=[chatbot, history_state, labels_box, tweets_box],
    )

demo.launch(share=True, show_error=True, theme=gr.themes.Soft())

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bfbe5e84539b50020e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# RAG Comparison: Standard vs Classification-Guided

In [ ]:
# RAG Comparison: Standard vs Classification-Guided
import time

test_queries = [
    "What kind of help do people need after the flood?",
    "Are there missing people after the earthquake?",
    "What infrastructure was damaged in the hurricane?",
    "Where can people donate for disaster relief?",
    "What evacuation routes are available?",
]

print("=" * 70)
print("RAG COMPARISON: Standard vs Classification-Guided")
print("=" * 70)
print(f"{'Query':<45} {'Standard':>10} {'Event-only':>12} {'Dual':>8}")
print("-" * 70)

for query in test_queries:
    labels = classify_query(query)

    results = {}
    for mode in ["none", "event", "dual"]:
        t0 = time.time()
        retrieved = retrieve(query, labels, top_k=5, filter_mode=mode)
        latency = time.time() - t0

        # Score = average similarity of retrieved tweets
        avg_score = sum(t["score"] for t in retrieved) / len(retrieved)
        results[mode] = {"avg_score": avg_score, "latency": latency}

    print(f"{query[:44]:<45} "
          f"{results['none']['avg_score']:>9.3f} "
          f"{results['event']['avg_score']:>11.3f} "
          f"{results['dual']['avg_score']:>8.3f}")

print("\nHigher similarity score = more relevant retrieved tweets")
print("\nLatency (seconds):")
print(f"  Standard RAG:              ~{results['none']['latency']:.3f}s")
print(f"  Event-filtered RAG:        ~{results['event']['latency']:.3f}s")
print(f"  Dual-filtered RAG (ours):  ~{results['dual']['latency']:.3f}s")

RAG COMPARISON: Standard vs Classification-Guided
Query                                           Standard   Event-only     Dual
----------------------------------------------------------------------
What kind of help do people need after the f      2.237       2.114    1.288


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Are there missing people after the earthquak      0.435      -1.971   -3.569
What infrastructure was damaged in the hurri     -3.265      -3.265   -3.265
Where can people donate for disaster relief?      2.262       2.532    0.842
What evacuation routes are available?            -3.201      -4.554   -5.727

Higher similarity score = more relevant retrieved tweets

Latency (seconds):
  Standard RAG:              ~0.043s
  Event-filtered RAG:        ~0.046s
  Dual-filtered RAG (ours):  ~0.050s
